Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import dask.dataframe as dd
from IPython.display import display

pd.set_option('display.max_columns', None)

In [ ]:
# Definir rango de fechas para Enero 2021 (no está disponible todo enero 2020)

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

df = []
df2 = []

In [ ]:
# Importar Enero 2021 - forma 1

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

Tiempo de carga: 2.61 s


In [ ]:
# Importar Enero 2021 - forma 2 (con Dask)

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df2.append(dd.read_csv(url, dtype={'Admin2': 'object'}, assume_missing=True))

# Combinar los DataFrames diarios en uno solo.
enero = dd.concat(df2, ignore_index=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

Tiempo de carga con Dask: 2.96 s


In [ ]:
# 1. Cargar y visualizar los primeros 5 registros

enero.head()

,FIPS,Admin2,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,Combined_Key,Incident_Rate,Case_Fatality_Ratio
0,NaN,<NA>,<NA>,Afghanistan,2021-01-02 05:22:33,33.93911,67.709953,52513.0,2201.0,41727.0,8585.0,Afghanistan,134.896578,4.191343
1,NaN,<NA>,<NA>,Albania,2021-01-02 05:22:33,41.15330,20.168300,58316.0,1181.0,33634.0,23501.0,Albania,2026.409062,2.025173
2,NaN,<NA>,<NA>,Algeria,2021-01-02 05:22:33,28.03390,1.659600,99897.0,2762.0,67395.0,29740.0,Algeria,227.809861,2.764848
3,NaN,<NA>,<NA>,Andorra,2021-01-02 05:22:33,42.50630,1.521800,8117.0,84.0,7463.0,570.0,Andorra,10505.403482,1.034865
4,NaN,<NA>,<NA>,Angola,2021-01-02 05:22:33,-11.20270,17.873900,17568.0,405.0,11146.0,6017.0,Angola,53.452981,2.305328


In [ ]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

enero_pd = enero.compute()

print('Filas en total: ', len(enero_pd))
print('Columnas en total: ', len(enero_pd.columns))

Filas en total:  124398
Columnas en total:  14


In [ ]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update            string[pyarrow]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [ ]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero = enero.assign(Last_Update=dd.to_datetime(enero['Last_Update'], errors='coerce'))

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update             datetime64[ns]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [ ]:
# 3

# Uso de memoria antes de conversión de tipos

enero_pd = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria = enero_pd.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame: {memoria:.2f} MB")

Memoria usada del DataFrame: 18.69 MB


In [ ]:
# 3

# Uso de memoria después de conversión de tipos

enero['Province_State'] = enero['Province_State'].astype('category')
enero['Country_Region'] = enero['Country_Region'].astype('category')
enero['Combined_Key'] = enero['Combined_Key'].astype('category')

enero_pdf = enero.compute()

# Verificar memoria usada por el DataFrame pandas resultante
memoria_optimizacion = enero_pdf.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Memoria usada del DataFrame tras optimización: {memoria_optimizacion:.2f} MB")

Memoria usada del DataFrame tras optimización: 13.08 MB


In [ ]:
print(f"Diferencia en uso de memoria: {(memoria - memoria_optimizacion):.2f} MB")

Diferencia en uso de memoria: 5.61 MB


In [ ]:
print(f"Diferencia en uso de memoria: {(-(memoria - memoria_optimizacion) * 100 / memoria):.0f}%")

Diferencia en uso de memoria: -30%


In [ ]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

enero.isnull().sum().compute()

province_state         5529
country_region            0
last_update              37
confirmed                 0
deaths                    0
recovered                 0
active                    0
incident_rate          2776
case_fatality_ratio    1484
active_cases              0
dtype: int64

In [ ]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(0)

,Province_State,Country_Region,Last_Update,Confirmed,Deaths,Recovered,Active,Incident_Rate,Case_Fatality_Ratio


In [ ]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(0)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases


In [ ]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
648,Alabama,United States,2021-01-02 05:22:33,4239.0,50.0,0.0,4189.0,7587.391935,1.179523


In [ ]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = dd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,<NA>,Afghanistan,2021-01-02,52513.0,2201.0,41727.0,8585.0,134.896578,4.191343


In [ ]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases
0,<NA>,Afghanistan,2021-01-02,52513.0,2201.0,41727.0,8585.0,134.896578,4.191343,8585.0


In [ ]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2021.csv e indicar su tamaño en MB.

try:
    enero.compute().to_csv('covid_clean_enero2021.csv')
except:
    os.remove('covid_clean_enero2021.csv')
    enero.compute().to_csv('covid_clean_enero2021.csv')

file_size = os.path.getsize('covid_clean_enero2021.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2021.csv es: {file_size:.2f} MB')

El tamaño del archivo covid_clean_enero2021.csv es: 12.22 MB
